In [ ]:
%%capture
!pip install openpyxl statsmodels yfinance
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
print("Libraries ready!")

In [ ]:
import yfinance as yf

# ── EDGAR ──────────────────────────────────────────────
edgar = pd.read_excel('edgar_panel_data.xlsx', sheet_name='Quarterly')
edgar['end'] = pd.to_datetime(edgar['end'])

# FIX: use available_from = end + 1 month to avoid look-ahead bias
# Q4 2023 (end = Dec 31) becomes available from Jan 1, 2024
edgar['available_from'] = (edgar['end'].dt.to_period('M') + 1).dt.to_timestamp()
print(f"EDGAR: {len(edgar)} rows, {edgar['Ticker'].nunique()} firms")

# ── FEAR & GREED (end-of-month + FIRST DIFFERENCE + distance) ──
fng_raw = pd.read_csv('fear_greed_index.csv')
fng_raw['date'] = pd.to_datetime(fng_raw['date'])
fng_raw = fng_raw.sort_values('date')
fng_raw['YearMonth'] = fng_raw['date'].dt.to_period('M')

# End-of-month value
fng_eom = fng_raw.groupby('YearMonth').last().reset_index()
fng_eom['Date'] = fng_eom['YearMonth'].dt.to_timestamp()
fng_eom = fng_eom[['Date', 'value']].rename(columns={'value': 'FearGreed'})
fng_eom['Date'] = fng_eom['Date'].dt.to_period('M').dt.to_timestamp()
fng_eom = fng_eom.sort_values('Date')

# Monthly change: FIRST DIFFERENCE (index points, not percentage)
fng_eom['FearGreed_change'] = fng_eom['FearGreed'].diff()

# Distance variables using daily rolling 252 trading days
fng_raw['rolling_max_252'] = fng_raw['value'].rolling(window=252, min_periods=1).max()
fng_raw['rolling_min_252'] = fng_raw['value'].rolling(window=252, min_periods=1).min()
fng_raw['FG_dist_high'] = (fng_raw['value'] / fng_raw['rolling_max_252']) - 1
fng_raw['FG_dist_low']  = (fng_raw['value'] / fng_raw['rolling_min_252']) - 1

# Take end-of-month values of distance measures
fng_dist = fng_raw.groupby('YearMonth').last().reset_index()
fng_dist['Date'] = fng_dist['YearMonth'].dt.to_timestamp()
fng_dist = fng_dist[['Date', 'FG_dist_high', 'FG_dist_low']]
fng_dist['Date'] = fng_dist['Date'].dt.to_period('M').dt.to_timestamp()

# Combine into one monthly dataframe
fng_monthly = fng_eom.merge(fng_dist, on='Date', how='left')
print(f"Fear & Greed: {len(fng_monthly)} months (end-of-month, first difference)")
print(fng_monthly[['Date', 'FearGreed', 'FearGreed_change']].head(5))

# ── STOCK RETURNS ───────────────────────────────────────
returns2 = pd.read_csv('sp500_monthly_returns.csv', index_col=0)
returns2.index = pd.to_datetime(returns2.index, utc=True).tz_localize(None)
returns2.index = returns2.index.to_period('M').to_timestamp()
print(f"Returns: {returns2.shape}")

# ── MARKET RETURN ───────────────────────────────────────
market = yf.download('^GSPC', start='2018-02-01', end='2025-05-02',
                     interval='1mo', auto_adjust=False, progress=False)
market_return = market['Adj Close'].pct_change().dropna()
market_return.index = market_return.index.to_period('M').to_timestamp()
market_return.name = 'MarketReturn'

# ── VIX ────────────────────────────────────────────────
vix = yf.download('^VIX', start='2018-02-01', end='2025-05-02',
                  interval='1mo', auto_adjust=False, progress=False)
vix_level = vix['Adj Close'].dropna()
vix_level.index = vix_level.index.to_period('M').to_timestamp()
vix_level.name = 'VIX'

print("All data loaded!")

EDGAR: 21249 rows, 500 firms
Fear & Greed: 88 months (end-of-month, first difference)
        Date  FearGreed  FearGreed_change
0 2018-02-01         41               NaN
1 2018-03-01         16             -25.0
2 2018-04-01         59              43.0
3 2018-05-01         25             -34.0
4 2018-06-01         22              -3.0
Returns: (88, 501)
All data loaded!


In [ ]:
print("Building stock return panel...")

# Reshape returns to long format
returns_long = returns2.stack().reset_index()
returns_long.columns = ['Date', 'Ticker', 'Return']
returns_long['Return'] = returns_long['Return'] / 100

# Add market return
market_df = market_return.reset_index()
market_df.columns = ['Date', 'MarketReturn']

# Add VIX level and change
vix_df = vix_level.reset_index()
vix_df.columns = ['Date', 'VIX']
vix_df = vix_df.sort_values('Date')
vix_df['VIX_change'] = vix_df['VIX'].pct_change()

returns_long = returns_long.merge(market_df, on='Date', how='left')
returns_long = returns_long.merge(vix_df, on='Date', how='left')

# Add excess return
returns_long['Return_ex'] = returns_long['Return'] - returns_long['MarketReturn']

# Add all Fear & Greed variables
returns_long = returns_long.merge(
    fng_monthly[['Date', 'FearGreed', 'FearGreed_change', 'FG_dist_high', 'FG_dist_low']],
    on='Date', how='left'
)

print(f"Panel: {len(returns_long)} firm-month observations")
print(f"Firms: {returns_long['Ticker'].nunique()}")
print(f"Months: {returns_long['Date'].nunique()}")

Building stock return panel...
Panel: 42705 firm-month observations
Firms: 501
Months: 87


In [ ]:
print("Merging EDGAR financial data...")

edgar_clean = edgar.dropna(subset=['available_from']).copy()
edgar_clean = edgar_clean.sort_values(['Ticker', 'available_from'])

returns_long = returns_long.sort_values(['Ticker', 'Date'])

panel_parts = []
for ticker in returns_long['Ticker'].unique():
    ret = returns_long[returns_long['Ticker'] == ticker].copy()
    fin = edgar_clean[edgar_clean['Ticker'] == ticker].copy()
    if len(ret) == 0 or len(fin) == 0:
        continue
    merged = pd.merge_asof(
        ret.sort_values('Date'),
        fin.sort_values('available_from'),
        left_on='Date',
        right_on='available_from',
        by='Ticker',
        direction='backward'
    )
    panel_parts.append(merged)

panel = pd.concat(panel_parts, ignore_index=True)
panel = panel.loc[:, ~panel.columns.duplicated()]
print(f"Panel after merge: {len(panel)} observations, {panel['Ticker'].nunique()} firms")

Merging EDGAR financial data...
Panel after merge: 42457 observations, 498 firms


In [ ]:
# Chunk 5: Construct derived variables
print("Constructing variables...")

# Get monthly stock prices for market cap
tickers_list = list(returns2.columns)
prices_raw = yf.download(tickers_list, start='2018-02-01', end='2025-05-02',
                         interval='1mo', auto_adjust=True, progress=False)['Close']
prices_raw.index = prices_raw.index.to_period('M').to_timestamp()
prices_long = prices_raw.stack().reset_index()
prices_long.columns = ['Date', 'Ticker', 'Price']
panel = panel.merge(prices_long, on=['Date', 'Ticker'], how='left')

# Firm size: TTM Revenue
edgar_sorted = edgar_clean.sort_values(['Ticker', 'end'])
edgar_sorted['TTM_Revenue'] = edgar_sorted.groupby('Ticker')['Revenue'].transform(
    lambda x: x.rolling(4, min_periods=4).sum()
)
edgar_sorted['lnRevenue'] = np.log(edgar_sorted['TTM_Revenue'].replace(0, np.nan))
panel = panel.merge(
    edgar_sorted[['Ticker', 'end', 'TTM_Revenue', 'lnRevenue']].dropna(),
    on=['Ticker', 'end'], how='left'
)

# Leverage
panel['TotalDebt'] = panel['LongTermDebt'].fillna(0) + panel['CurrentDebt'].fillna(0)
panel['Leverage'] = panel['TotalDebt'] / panel['TotalAssets']
panel['Leverage'] = panel['Leverage'].replace([np.inf, -np.inf], np.nan).clip(0, 2)

# Book-to-Market: drop firms with negative book equity (standard in asset pricing)
panel['MarketCap'] = panel['Price'] * panel['SharesOutstanding']
panel = panel[panel['TotalEquity'] > 0]  # drop negative book equity firms
panel['BookToMarket'] = panel['TotalEquity'] / panel['MarketCap']
panel['BookToMarket'] = panel['BookToMarket'].replace([np.inf, -np.inf], np.nan).clip(0, 10)

# Momentum controls
panel = panel.sort_values(['Ticker', 'Date'])
panel['Ret_1m'] = panel.groupby('Ticker')['Return'].shift(1)
panel['Ret_6m'] = panel.groupby('Ticker')['Return'].transform(
    lambda x: x.shift(2).rolling(6, min_periods=4).sum()
)

# Year and clustering variables
panel['Year'] = panel['Date'].dt.year
panel['YearMonth_int'] = panel['Date'].dt.year * 100 + panel['Date'].dt.month
panel['Ticker_int'] = panel['Ticker'].astype('category').cat.codes

print("Variables constructed!")
print(f"Panel shape: {panel.shape}")
key_vars = ['Return_ex', 'FearGreed_change', 'FG_dist_high', 'FG_dist_low',
            'VIX', 'VIX_change', 'lnRevenue', 'Leverage', 'BookToMarket',
            'Ret_1m', 'Ret_6m']
print(panel[key_vars].notna().sum())

Constructing variables...
Variables constructed!
Panel shape: (39426, 50)
Return_ex           39426
FearGreed_change    39426
FG_dist_high        39426
FG_dist_low         39426
VIX                 39426
VIX_change          39426
lnRevenue           38693
Leverage            39403
BookToMarket        38797
Ret_1m              38938
Ret_6m              36992
dtype: int64


In [ ]:
# Table 2: Descriptive Statistics
desc_vars = ['Return_ex', 'FearGreed_change', 'FG_dist_high', 'FG_dist_low',
             'VIX', 'VIX_change', 'lnRevenue', 'Leverage', 'BookToMarket',
             'Ret_1m', 'Ret_6m']

desc = panel[desc_vars].describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
desc = desc.round(4)
print("Table 2: Descriptive Statistics")
print(desc.to_string())

Table 2: Descriptive Statistics
                    count     mean      std      min      25%      50%      75%      max
Return_ex         39426.0   0.0023   0.0805  -0.6616  -0.0435  -0.0006   0.0437   1.5687
FearGreed_change  39426.0   0.2947  21.5364 -60.0000 -12.0000   0.0000  17.0000  54.0000
FG_dist_high      39426.0  -0.4116   0.2472  -0.8690  -0.6500  -0.3810  -0.1915   0.0000
FG_dist_low       39426.0   3.2725   2.4493   0.4474   1.1250   2.5000   5.0000  10.0000
VIX               39426.0  20.1783   7.2247  12.1200  15.0800  18.2400  24.4600  53.5400
VIX_change        39426.0   0.0333   0.2763  -0.4590  -0.1744  -0.0291   0.1774   1.1290
lnRevenue         38693.0  23.1471   1.2955  14.9108  22.3059  23.1254  23.8756  27.2246
Leverage          39403.0   0.2499   0.1771   0.0000   0.1027   0.2528   0.3736   2.0000
BookToMarket      38797.0   0.4754   0.6259   0.0000   0.1562   0.3190   0.6044  10.0000
Ret_1m            38938.0   0.0127   0.0955  -0.7867  -0.0421   0.0117   0.063

In [ ]:
print("Running regressions with Year FE and double-clustered SEs...")

import statsmodels.formula.api as smf
import scipy.stats as stats

panel = panel.sort_values(['Ticker', 'Date'])

panel['Return_ex_fwd1']  = panel.groupby('Ticker')['Return_ex'].shift(-1)
panel['Return_ex_fwd3']  = panel.groupby('Ticker')['Return_ex'].shift(-3)
panel['Return_ex_fwd12'] = panel.groupby('Ticker')['Return_ex'].shift(-12)

panel['FG_x_lnRevenue']    = panel['FearGreed_change'] * panel['lnRevenue']
panel['FG_x_BookToMarket'] = panel['FearGreed_change'] * panel['BookToMarket']

for horizon, y_var in [
    ('1 month', 'Return_ex_fwd1'),
    ('3 months', 'Return_ex_fwd3'),
    ('12 months', 'Return_ex_fwd12')
]:
    reg_data = panel.dropna(subset=[
        y_var, 'FearGreed_change', 'FG_dist_high', 'FG_dist_low',
        'VIX', 'VIX_change', 'lnRevenue', 'Leverage',
        'BookToMarket', 'Ret_1m', 'Ret_6m',
        'FG_x_lnRevenue', 'FG_x_BookToMarket',
        'Ticker_int', 'YearMonth_int'
    ]).copy()

    formula = f"""
    {y_var} ~ FearGreed_change + FG_dist_high + FG_dist_low +
    VIX + VIX_change +
    lnRevenue + Leverage + BookToMarket +
    Ret_1m + Ret_6m +
    FG_x_lnRevenue + FG_x_BookToMarket +
    C(Year)
    """

    model_base  = smf.ols(formula, data=reg_data).fit()
    model_firm  = smf.ols(formula, data=reg_data).fit(
        cov_type='cluster',
        cov_kwds={'groups': reg_data['Ticker_int'].values}
    )
    model_month = smf.ols(formula, data=reg_data).fit(
        cov_type='cluster',
        cov_kwds={'groups': reg_data['YearMonth_int'].values}
    )

    V_double  = model_firm.cov_params() + model_month.cov_params() - model_base.cov_params()
    se_double = np.sqrt(np.diag(V_double))
    t_stats   = model_base.params / se_double
    dof       = min(reg_data['Ticker_int'].nunique(), reg_data['YearMonth_int'].nunique()) - 1
    p_double  = 2 * stats.t.sf(np.abs(t_stats), df=dof)

    print(f"\n{'='*60}")
    print(f"HORIZON: {horizon}")
    print(f"Observations: {len(reg_data)}, Firms: {reg_data['Ticker_int'].nunique()}")
    print(f"R-squared: {model_base.rsquared:.4f}")
    print(f"\n{'Variable':<25} {'Coef':>12} {'P-value':>10}")
    print("-"*50)
    for var in ['FearGreed_change', 'FG_dist_high', 'FG_dist_low',
                'VIX', 'VIX_change', 'lnRevenue', 'Leverage', 'BookToMarket',
                'Ret_1m', 'Ret_6m', 'FG_x_lnRevenue', 'FG_x_BookToMarket', 'Intercept']:
        if var in model_base.params.index:
            coef = model_base.params[var]
            idx  = list(model_base.params.index).index(var)
            pval = p_double[idx]
            stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
            print(f"{var:<25} {coef:>12.6f} {pval:>10.4f} {stars}")

print("\nDone!")

Running regressions with Year FE and double-clustered SEs...

HORIZON: 1 month
Observations: 35357, Firms: 473
R-squared: 0.0088

Variable                          Coef    P-value
--------------------------------------------------
FearGreed_change              0.000834     0.3118 
FG_dist_high                  0.022345     0.0206 **
FG_dist_low                  -0.001049     0.3609 
VIX                           0.000262     0.5608 
VIX_change                    0.000816     0.9365 
lnRevenue                    -0.001218     0.1151 
Leverage                     -0.003263     0.5084 
BookToMarket                  0.003820     0.0544 *
Ret_1m                       -0.002672     0.8915 
Ret_6m                        0.009735     0.2394 
FG_x_lnRevenue               -0.000042     0.2415 
FG_x_BookToMarket            -0.000054     0.6494 
Intercept                     0.042905     0.0429 **

HORIZON: 3 months
Observations: 34412, Firms: 473
R-squared: 0.0140

Variable                       

In [ ]:
print("Fear & Greed change summary:")
print(fng_monthly[['FearGreed', 'FearGreed_change']].describe())
print("\nFirst 10 rows:")
print(fng_monthly[['Date', 'FearGreed', 'FearGreed_change']].head(10))

Fear & Greed change summary:
       FearGreed  FearGreed_change
count  88.000000         87.000000
mean   47.272727          0.298851
std    21.380906         21.649488
min    11.000000        -60.000000
25%    28.000000        -12.000000
50%    49.500000          0.000000
75%    63.250000         17.000000
max    95.000000         54.000000

First 10 rows:
        Date  FearGreed  FearGreed_change
0 2018-02-01         41               NaN
1 2018-03-01         16             -25.0
2 2018-04-01         59              43.0
3 2018-05-01         25             -34.0
4 2018-06-01         22              -3.0
5 2018-07-01         48              26.0
6 2018-08-01         17             -31.0
7 2018-09-01         34              17.0
8 2018-10-01         32              -2.0
9 2018-11-01         19             -13.0
